In [31]:
import os

In [32]:
%pwd

'e:\\'

In [33]:
os.chdir("../")

In [34]:
%pwd

'e:\\'

In [35]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [ ]:
from dataclasses import dataclass
from pathlib import Path
@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: str
    max_input_length: int
    max_target_length: int

In [ ]:
from textSummarizer.entity import DataTransformationConfig
from textSummarizer.utils.common import read_yaml, create_directories
from pathlib import Path


class ConfigurationManager:
    def __init__(
        self,
        config_filepath: Path = Path("config/config.yaml"),
        params_filepath: Path = Path("params.yaml"),
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            tokenizer_name=config.tokenizer_name,
            max_input_length=config.max_input_length,
            max_target_length=config.max_target_length,
        )

        return data_transformation_config

In [ ]:
import os
from transformers import T5Tokenizer
from textSummarizer.logging import logger
from datasets import load_from_disk

In [ ]:
import os
from transformers import T5Tokenizer
from textSummarizer.logging import logger
from datasets import load_from_disk
from textSummarizer.entity import DataTransformationConfig


class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = T5Tokenizer.from_pretrained(self.config.tokenizer_name)

    def convert_examples_to_features(self, example_batch):

        # ✅ Add T5 prefix
        inputs = ["summarize: " + dialogue for dialogue in example_batch["dialogue"]]

        # Tokenize inputs
        model_inputs = self.tokenizer(
            inputs,
            max_length=self.config.max_input_length,
            truncation=True,
            padding="max_length"
        )

        # Tokenize targets
        labels = self.tokenizer(
            example_batch["summary"],
            max_length=self.config.max_target_length,
            truncation=True,
            padding="max_length"
        )

        # ✅ Replace padding token id with -100
        labels_ids = labels["input_ids"]
        labels_ids = [
            [(token if token != self.tokenizer.pad_token_id else -100) for token in label]
            for label in labels_ids
        ]

        model_inputs["labels"] = labels_ids

        return model_inputs

    def convert(self):
        logger.info("Loading dataset from disk...")
        dataset_samsum = load_from_disk(self.config.data_path)

        logger.info("Starting T5 data transformation...")

        dataset_samsum_pt = dataset_samsum.map(
            self.convert_examples_to_features,
            batched=True,
            remove_columns=dataset_samsum["train"].column_names  # cleaner dataset
        )

        output_path = os.path.join(self.config.root_dir, "samsum_dataset")

        dataset_samsum_pt.save_to_disk(output_path)

        logger.info(f"Transformed dataset saved at {output_path}")

In [ ]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:    
    raise e


[2026-03-24 21:50:16,225: INFO: common]: yaml file: E:\Text-Summarizer\config\config.yaml loaded successfully
[2026-03-24 21:50:16,225: INFO: common]: yaml file: E:\Text-Summarizer\params.yaml loaded successfully
[2026-03-24 21:50:16,225: INFO: common]: created directory at: artifacts
[2026-03-24 21:50:16,231: INFO: common]: created directory at: artifacts/data_transformation
[2026-03-24 21:50:16,710: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-03-24 21:50:17,006: INFO: _client]: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"
[2026-03-24 21:50:17,048: INFO: _client]: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"


e:\Text-Summarizer\textS\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--google--pegasus-cnn_dailymail. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


[2026-03-24 21:50:17,378: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
[2026-03-24 21:50:17,399: INFO: _client]: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK"
[2026-03-24 21:50:17,441: INFO: _client]: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK"
[2026-03-24 21:50:17,706: INFO: _client]: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
[2026-03-24 21:50:17,960: INFO: _client]: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main?recursive=true&expand=false "HTTP/1

[2026-03-24 21:50:20,102: WARNING: _http]: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[2026-03-24 21:50:20,124: INFO: _client]: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/special_tokens_map.json "HTTP/1.1 200 OK"
[2026-03-24 21:50:20,156: INFO: _client]: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/special_tokens_map.json "HTTP/1.1 200 OK"
[2026-03-24 21:50:20,459: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"


Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 90499.87 examples/s]
